In [1]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

RANDOM_STATE = 42

TRAIN_PATH = "train.csv"
TEST_PATH = "test-unlabelled.csv"

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Columnas finales:", train.columns[-5:].tolist())

X = train.drop(columns=["class"])
y = train["class"]

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

results = []

def evaluate_model(name, model, notes=""):
    """
    Evalúa un modelo con:
    - accuracy en todo el training set
    - accuracy media con 5-fold cross-validation
    """
    model.fit(X, y)
    train_pred = model.predict(X)
    train_acc = accuracy_score(y, train_pred)

    cv_scores = cross_val_score(
        model,
        X,
        y,
        cv=cv,
        scoring="accuracy",
        n_jobs=-1
    )

    row = {
        "experiment": name,
        "train_accuracy": train_acc,
        "cv_mean": cv_scores.mean(),
        "cv_std": cv_scores.std(),
        "notes": notes
    }
    results.append(row)

    print(f"Experimento: {name}")
    print(f"Train accuracy: {train_acc:.4f}")
    print(f"5-fold CV accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
    print(f"Notas: {notes}")

    return model, cv_scores

Train shape: (1000, 610)
Test shape: (2000, 610)
Columnas finales: ['nwpqkzh', 'yjsknnh', 'qtfvayr', 'kaxienf', 'class']


In [2]:
print("Dimensiones X:", X.shape)
print("\nDistribución de clases:")
print(y.value_counts())
print("\nProporción de clases:")
print(y.value_counts(normalize=True))

print("\nValores nulos en train:", train.isna().sum().sum())
print("Valores nulos en test:", test.isna().sum().sum())

X.describe().T.head()

Dimensiones X: (1000, 609)

Distribución de clases:
class
A    347
B    328
C    325
Name: count, dtype: int64

Proporción de clases:
class
A    0.347
B    0.328
C    0.325
Name: proportion, dtype: float64

Valores nulos en train: 0
Valores nulos en test: 2000


,count,mean,std,min,25%,50%,75%,max
lnauxax,1000.0,6.962000,2.654355,0.000000,5.000000,7.000000,9.000000,16.000000
frpcvrs,1000.0,3.085164,2.429008,0.025600,1.332050,2.510294,4.110204,16.984291
vhztoil,1000.0,4.888311,3.110253,0.216418,2.583480,4.204823,6.614121,19.568443
jvvthfj,1000.0,-0.407401,8.232834,-109.570960,-2.187783,0.787451,2.583759,38.072782
oniwlqr,1000.0,3.015783,2.457959,0.010424,1.207115,2.347860,4.295479,20.342250


## Gaussian Naive Bayes baseline

In [3]:
from sklearn.naive_bayes import GaussianNB

nb_baseline = GaussianNB()

model_nb_baseline, scores_nb_baseline = evaluate_model(
    name="NB-01 GaussianNB baseline",
    model=nb_baseline,
    notes="Baseline sin preprocesamiento. Sirve para medir el rendimiento inicial de Naive Bayes."
)

Experimento: NB-01 GaussianNB baseline
Train accuracy: 0.6190
5-fold CV accuracy: 0.5390 (+/- 0.0377)
Notas: Baseline sin preprocesamiento. Sirve para medir el rendimiento inicial de Naive Bayes.


## StandardScaler + GaussianNB

In [4]:
from sklearn.preprocessing import StandardScaler

nb_scaled = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", GaussianNB())
])

model_nb_scaled, scores_nb_scaled = evaluate_model(
    name="NB-02 StandardScaler + GaussianNB",
    model=nb_scaled,
    notes="Prueba de escalado. Se prueba para comprobar si normalizar las variables afecta al rendimiento de GaussianNB."
)

Experimento: NB-02 StandardScaler + GaussianNB
Train accuracy: 0.6190
5-fold CV accuracy: 0.5390 (+/- 0.0377)
Notas: Prueba de escalado. Se prueba para comprobar si normalizar las variables afecta al rendimiento de GaussianNB.


In [5]:
pd.DataFrame(results)

,experiment,train_accuracy,cv_mean,cv_std,notes
0,NB-01 GaussianNB baseline,0.619,0.539,0.037736,Baseline sin preprocesamiento. Sirve para medi...
1,NB-02 StandardScaler + GaussianNB,0.619,0.539,0.037736,Prueba de escalado. Se prueba para comprobar s...


## GaussianNB con selección de variables

In [6]:
from sklearn.feature_selection import SelectKBest, f_classif

k_values = [5, 10, 20, 40, 60, 100, 200, 300]

nb_kbest_models = {}

for k in k_values:
    nb_kbest = Pipeline([
        ("select", SelectKBest(score_func=f_classif, k=k)),
        ("classifier", GaussianNB())
    ])
    
    model, scores = evaluate_model(
        name=f"NB-03 SelectKBest k={k} + GaussianNB",
        model=nb_kbest,
        notes=f"Selección de las {k} variables más relacionadas individualmente con la clase mediante ANOVA F-test."
    )
    
    nb_kbest_models[k] = {
        "model": model,
        "scores": scores
    }

Experimento: NB-03 SelectKBest k=5 + GaussianNB
Train accuracy: 0.6250
5-fold CV accuracy: 0.6210 (+/- 0.0285)
Notas: Selección de las 5 variables más relacionadas individualmente con la clase mediante ANOVA F-test.
Experimento: NB-03 SelectKBest k=10 + GaussianNB
Train accuracy: 0.6410
5-fold CV accuracy: 0.6420 (+/- 0.0211)
Notas: Selección de las 10 variables más relacionadas individualmente con la clase mediante ANOVA F-test.
Experimento: NB-03 SelectKBest k=20 + GaussianNB
Train accuracy: 0.6260
5-fold CV accuracy: 0.6380 (+/- 0.0196)
Notas: Selección de las 20 variables más relacionadas individualmente con la clase mediante ANOVA F-test.
Experimento: NB-03 SelectKBest k=40 + GaussianNB
Train accuracy: 0.6110
5-fold CV accuracy: 0.6150 (+/- 0.0228)
Notas: Selección de las 40 variables más relacionadas individualmente con la clase mediante ANOVA F-test.
Experimento: NB-03 SelectKBest k=60 + GaussianNB
Train accuracy: 0.5870
5-fold CV accuracy: 0.5950 (+/- 0.0152)
Notas: Selección d

In [7]:
results_df = pd.DataFrame(results)

results_df.sort_values("cv_mean", ascending=False)

,experiment,train_accuracy,cv_mean,cv_std,notes
3,NB-03 SelectKBest k=10 + GaussianNB,0.641,0.642,0.021119,Selección de las 10 variables más relacionadas...
4,NB-03 SelectKBest k=20 + GaussianNB,0.626,0.638,0.019647,Selección de las 20 variables más relacionadas...
2,NB-03 SelectKBest k=5 + GaussianNB,0.625,0.621,0.028531,Selección de las 5 variables más relacionadas ...
5,NB-03 SelectKBest k=40 + GaussianNB,0.611,0.615,0.022804,Selección de las 40 variables más relacionadas...
6,NB-03 SelectKBest k=60 + GaussianNB,0.587,0.595,0.015166,Selección de las 60 variables más relacionadas...
7,NB-03 SelectKBest k=100 + GaussianNB,0.571,0.580,0.018166,Selección de las 100 variables más relacionada...
8,NB-03 SelectKBest k=200 + GaussianNB,0.585,0.566,0.022450,Selección de las 200 variables más relacionada...
9,NB-03 SelectKBest k=300 + GaussianNB,0.591,0.559,0.023537,Selección de las 300 variables más relacionada...
0,NB-01 GaussianNB baseline,0.619,0.539,0.037736,Baseline sin preprocesamiento. Sirve para medi...
1,NB-02 StandardScaler + GaussianNB,0.619,0.539,0.037736,Prueba de escalado. Se prueba para comprobar s...


In [8]:
k_values_fine = [6, 8, 10, 12, 15, 18, 20, 25, 30]

nb_kbest_fine_models = {}

for k in k_values_fine:
    nb_kbest = Pipeline([
        ("select", SelectKBest(score_func=f_classif, k=k)),
        ("classifier", GaussianNB())
    ])
    
    model, scores = evaluate_model(
        name=f"NB-04 Fine SelectKBest k={k} + GaussianNB",
        model=nb_kbest,
        notes=f"Ajuste fino de selección de variables alrededor del mejor valor previo. k={k}."
    )
    
    nb_kbest_fine_models[k] = {
        "model": model,
        "scores": scores
    }

Experimento: NB-04 Fine SelectKBest k=6 + GaussianNB
Train accuracy: 0.6220
5-fold CV accuracy: 0.6160 (+/- 0.0357)
Notas: Ajuste fino de selección de variables alrededor del mejor valor previo. k=6.
Experimento: NB-04 Fine SelectKBest k=8 + GaussianNB
Train accuracy: 0.6220
5-fold CV accuracy: 0.6240 (+/- 0.0235)
Notas: Ajuste fino de selección de variables alrededor del mejor valor previo. k=8.
Experimento: NB-04 Fine SelectKBest k=10 + GaussianNB
Train accuracy: 0.6410
5-fold CV accuracy: 0.6420 (+/- 0.0211)
Notas: Ajuste fino de selección de variables alrededor del mejor valor previo. k=10.
Experimento: NB-04 Fine SelectKBest k=12 + GaussianNB
Train accuracy: 0.6400
5-fold CV accuracy: 0.6400 (+/- 0.0235)
Notas: Ajuste fino de selección de variables alrededor del mejor valor previo. k=12.
Experimento: NB-04 Fine SelectKBest k=15 + GaussianNB
Train accuracy: 0.6360
5-fold CV accuracy: 0.6400 (+/- 0.0247)
Notas: Ajuste fino de selección de variables alrededor del mejor valor previo. 

In [9]:
results_df = pd.DataFrame(results)
results_df.sort_values("cv_mean", ascending=False).head(15)

,experiment,train_accuracy,cv_mean,cv_std,notes
3,NB-03 SelectKBest k=10 + GaussianNB,0.641,0.642,0.021119,Selección de las 10 variables más relacionadas...
12,NB-04 Fine SelectKBest k=10 + GaussianNB,0.641,0.642,0.021119,Ajuste fino de selección de variables alrededo...
13,NB-04 Fine SelectKBest k=12 + GaussianNB,0.640,0.640,0.023452,Ajuste fino de selección de variables alrededo...
14,NB-04 Fine SelectKBest k=15 + GaussianNB,0.636,0.640,0.024698,Ajuste fino de selección de variables alrededo...
16,NB-04 Fine SelectKBest k=20 + GaussianNB,0.626,0.638,0.019647,Ajuste fino de selección de variables alrededo...
4,NB-03 SelectKBest k=20 + GaussianNB,0.626,0.638,0.019647,Selección de las 20 variables más relacionadas...
15,NB-04 Fine SelectKBest k=18 + GaussianNB,0.620,0.637,0.016000,Ajuste fino de selección de variables alrededo...
17,NB-04 Fine SelectKBest k=25 + GaussianNB,0.634,0.635,0.016125,Ajuste fino de selección de variables alrededo...
11,NB-04 Fine SelectKBest k=8 + GaussianNB,0.622,0.624,0.023537,Ajuste fino de selección de variables alrededo...
2,NB-03 SelectKBest k=5 + GaussianNB,0.625,0.621,0.028531,Selección de las 5 variables más relacionadas ...


## SelectKBest + discretización + CategoricalNB

In [10]:
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.naive_bayes import CategoricalNB

discretization_results = {}

n_bins_values = [3, 5, 10]
strategies = ["uniform", "quantile"]

for n_bins in n_bins_values:
    for strategy in strategies:
        nb_discrete = Pipeline([
            ("select", SelectKBest(score_func=f_classif, k=10)),
            ("discretizer", KBinsDiscretizer(
                n_bins=n_bins,
                encode="ordinal",
                strategy=strategy
            )),
            ("classifier", CategoricalNB(alpha=1.0))
        ])
        
        model, scores = evaluate_model(
            name=f"NB-05 SelectKBest k=10 + KBins {strategy} bins={n_bins} + CategoricalNB",
            model=nb_discrete,
            notes=f"Discretización de las 10 mejores variables con {n_bins} bins y estrategia {strategy}, seguida de CategoricalNB."
        )
        
        discretization_results[(n_bins, strategy)] = {
            "model": model,
            "scores": scores
        }

Experimento: NB-05 SelectKBest k=10 + KBins uniform bins=3 + CategoricalNB
Train accuracy: 0.4010
5-fold CV accuracy: 0.4940 (+/- 0.0859)
Notas: Discretización de las 10 mejores variables con 3 bins y estrategia uniform, seguida de CategoricalNB.
Experimento: NB-05 SelectKBest k=10 + KBins quantile bins=3 + CategoricalNB
Train accuracy: 0.7500
5-fold CV accuracy: 0.7420 (+/- 0.0204)
Notas: Discretización de las 10 mejores variables con 3 bins y estrategia quantile, seguida de CategoricalNB.
Experimento: NB-05 SelectKBest k=10 + KBins uniform bins=5 + CategoricalNB
Train accuracy: 0.7370
5-fold CV accuracy: 0.7220 (+/- 0.0238)
Notas: Discretización de las 10 mejores variables con 5 bins y estrategia uniform, seguida de CategoricalNB.
Experimento: NB-05 SelectKBest k=10 + KBins quantile bins=5 + CategoricalNB
Train accuracy: 0.7730
5-fold CV accuracy: 0.7650 (+/- 0.0362)
Notas: Discretización de las 10 mejores variables con 5 bins y estrategia quantile, seguida de CategoricalNB.
Experime

In [11]:
results_df = pd.DataFrame(results)
results_df.sort_values("cv_mean", ascending=False).head(20)

,experiment,train_accuracy,cv_mean,cv_std,notes
24,NB-05 SelectKBest k=10 + KBins quantile bins=1...,0.794,0.771,0.022000,Discretización de las 10 mejores variables con...
22,NB-05 SelectKBest k=10 + KBins quantile bins=5...,0.773,0.765,0.036194,Discretización de las 10 mejores variables con...
20,NB-05 SelectKBest k=10 + KBins quantile bins=3...,0.750,0.742,0.020396,Discretización de las 10 mejores variables con...
23,NB-05 SelectKBest k=10 + KBins uniform bins=10...,0.761,0.741,0.022000,Discretización de las 10 mejores variables con...
21,NB-05 SelectKBest k=10 + KBins uniform bins=5 ...,0.737,0.722,0.023791,Discretización de las 10 mejores variables con...
12,NB-04 Fine SelectKBest k=10 + GaussianNB,0.641,0.642,0.021119,Ajuste fino de selección de variables alrededo...
3,NB-03 SelectKBest k=10 + GaussianNB,0.641,0.642,0.021119,Selección de las 10 variables más relacionadas...
13,NB-04 Fine SelectKBest k=12 + GaussianNB,0.640,0.640,0.023452,Ajuste fino de selección de variables alrededo...
14,NB-04 Fine SelectKBest k=15 + GaussianNB,0.636,0.640,0.024698,Ajuste fino de selección de variables alrededo...
4,NB-03 SelectKBest k=20 + GaussianNB,0.626,0.638,0.019647,Selección de las 20 variables más relacionadas...


In [12]:
k_values_cat = [8, 10, 12, 15, 20]
n_bins_values_cat = [5, 8, 10, 12, 15, 20]

cat_tuning_results = {}

for k in k_values_cat:
    for n_bins in n_bins_values_cat:
        nb_cat = Pipeline([
            ("select", SelectKBest(score_func=f_classif, k=k)),
            ("discretizer", KBinsDiscretizer(
                n_bins=n_bins,
                encode="ordinal",
                strategy="quantile"
            )),
            ("classifier", CategoricalNB(alpha=1.0))
        ])
        
        model, scores = evaluate_model(
            name=f"NB-06 SelectKBest k={k} + quantile bins={n_bins} + CategoricalNB",
            model=nb_cat,
            notes=f"Ajuste de CategoricalNB con selección de {k} variables y discretización quantile con {n_bins} bins."
        )
        
        cat_tuning_results[(k, n_bins)] = {
            "model": model,
            "scores": scores
        }

Experimento: NB-06 SelectKBest k=8 + quantile bins=5 + CategoricalNB
Train accuracy: 0.7570
5-fold CV accuracy: 0.7480 (+/- 0.0333)
Notas: Ajuste de CategoricalNB con selección de 8 variables y discretización quantile con 5 bins.
Experimento: NB-06 SelectKBest k=8 + quantile bins=8 + CategoricalNB
Train accuracy: 0.7750
5-fold CV accuracy: 0.7580 (+/- 0.0172)
Notas: Ajuste de CategoricalNB con selección de 8 variables y discretización quantile con 8 bins.
Experimento: NB-06 SelectKBest k=8 + quantile bins=10 + CategoricalNB
Train accuracy: 0.7840
5-fold CV accuracy: 0.7560 (+/- 0.0174)
Notas: Ajuste de CategoricalNB con selección de 8 variables y discretización quantile con 10 bins.
Experimento: NB-06 SelectKBest k=8 + quantile bins=12 + CategoricalNB
Train accuracy: 0.7820
5-fold CV accuracy: 0.7590 (+/- 0.0169)
Notas: Ajuste de CategoricalNB con selección de 8 variables y discretización quantile con 12 bins.
Experimento: NB-06 SelectKBest k=8 + quantile bins=15 + CategoricalNB
Train 

In [13]:
results_df = pd.DataFrame(results)
results_df.sort_values("cv_mean", ascending=False).head(25)

,experiment,train_accuracy,cv_mean,cv_std,notes
52,NB-06 SelectKBest k=20 + quantile bins=12 + Ca...,0.816,0.795,0.008944,Ajuste de CategoricalNB con selección de 20 va...
51,NB-06 SelectKBest k=20 + quantile bins=10 + Ca...,0.797,0.787,0.014353,Ajuste de CategoricalNB con selección de 20 va...
50,NB-06 SelectKBest k=20 + quantile bins=8 + Cat...,0.800,0.783,0.009274,Ajuste de CategoricalNB con selección de 20 va...
54,NB-06 SelectKBest k=20 + quantile bins=20 + Ca...,0.833,0.782,0.024207,Ajuste de CategoricalNB con selección de 20 va...
53,NB-06 SelectKBest k=20 + quantile bins=15 + Ca...,0.821,0.781,0.017146,Ajuste de CategoricalNB con selección de 20 va...
41,NB-06 SelectKBest k=12 + quantile bins=15 + Ca...,0.808,0.776,0.020347,Ajuste de CategoricalNB con selección de 12 va...
35,NB-06 SelectKBest k=10 + quantile bins=15 + Ca...,0.802,0.776,0.029394,Ajuste de CategoricalNB con selección de 10 va...
39,NB-06 SelectKBest k=12 + quantile bins=10 + Ca...,0.803,0.776,0.022226,Ajuste de CategoricalNB con selección de 12 va...
49,NB-06 SelectKBest k=20 + quantile bins=5 + Cat...,0.779,0.775,0.016125,Ajuste de CategoricalNB con selección de 20 va...
37,NB-06 SelectKBest k=12 + quantile bins=5 + Cat...,0.780,0.774,0.032156,Ajuste de CategoricalNB con selección de 12 va...


## Expandir seleccion de variables

In [14]:
k_values_cat_expanded = [20, 25, 30, 40, 60]
n_bins_values_cat_expanded = [8, 10, 12, 15]

cat_expanded_results = {}

for k in k_values_cat_expanded:
    for n_bins in n_bins_values_cat_expanded:
        nb_cat = Pipeline([
            ("select", SelectKBest(score_func=f_classif, k=k)),
            ("discretizer", KBinsDiscretizer(
                n_bins=n_bins,
                encode="ordinal",
                strategy="quantile"
            )),
            ("classifier", CategoricalNB(alpha=1.0))
        ])
        
        model, scores = evaluate_model(
            name=f"NB-07 Expanded SelectKBest k={k} + quantile bins={n_bins} + CategoricalNB",
            model=nb_cat,
            notes=f"Ampliación de búsqueda para CategoricalNB: {k} variables y {n_bins} bins quantile."
        )
        
        cat_expanded_results[(k, n_bins)] = {
            "model": model,
            "scores": scores
        }

Experimento: NB-07 Expanded SelectKBest k=20 + quantile bins=8 + CategoricalNB
Train accuracy: 0.8000
5-fold CV accuracy: 0.7830 (+/- 0.0093)
Notas: Ampliación de búsqueda para CategoricalNB: 20 variables y 8 bins quantile.
Experimento: NB-07 Expanded SelectKBest k=20 + quantile bins=10 + CategoricalNB
Train accuracy: 0.7970
5-fold CV accuracy: 0.7870 (+/- 0.0144)
Notas: Ampliación de búsqueda para CategoricalNB: 20 variables y 10 bins quantile.
Experimento: NB-07 Expanded SelectKBest k=20 + quantile bins=12 + CategoricalNB
Train accuracy: 0.8160
5-fold CV accuracy: 0.7950 (+/- 0.0089)
Notas: Ampliación de búsqueda para CategoricalNB: 20 variables y 12 bins quantile.
Experimento: NB-07 Expanded SelectKBest k=20 + quantile bins=15 + CategoricalNB
Train accuracy: 0.8210
5-fold CV accuracy: 0.7810 (+/- 0.0171)
Notas: Ampliación de búsqueda para CategoricalNB: 20 variables y 15 bins quantile.
Experimento: NB-07 Expanded SelectKBest k=25 + quantile bins=8 + CategoricalNB
Train accuracy: 0.8

In [15]:
results_df = pd.DataFrame(results)
results_df.sort_values("cv_mean", ascending=False).head(30)

,experiment,train_accuracy,cv_mean,cv_std,notes
74,NB-07 Expanded SelectKBest k=60 + quantile bin...,0.859,0.805,0.013416,Ampliación de búsqueda para CategoricalNB: 60 ...
72,NB-07 Expanded SelectKBest k=60 + quantile bin...,0.843,0.803,0.015684,Ampliación de búsqueda para CategoricalNB: 60 ...
73,NB-07 Expanded SelectKBest k=60 + quantile bin...,0.849,0.801,0.007348,Ampliación de búsqueda para CategoricalNB: 60 ...
71,NB-07 Expanded SelectKBest k=60 + quantile bin...,0.839,0.800,0.006325,Ampliación de búsqueda para CategoricalNB: 60 ...
59,NB-07 Expanded SelectKBest k=25 + quantile bin...,0.815,0.799,0.016852,Ampliación de búsqueda para CategoricalNB: 25 ...
70,NB-07 Expanded SelectKBest k=40 + quantile bin...,0.843,0.797,0.014697,Ampliación de búsqueda para CategoricalNB: 40 ...
52,NB-06 SelectKBest k=20 + quantile bins=12 + Ca...,0.816,0.795,0.008944,Ajuste de CategoricalNB con selección de 20 va...
57,NB-07 Expanded SelectKBest k=20 + quantile bin...,0.816,0.795,0.008944,Ampliación de búsqueda para CategoricalNB: 20 ...
62,NB-07 Expanded SelectKBest k=25 + quantile bin...,0.828,0.795,0.021213,Ampliación de búsqueda para CategoricalNB: 25 ...
64,NB-07 Expanded SelectKBest k=30 + quantile bin...,0.827,0.794,0.017436,Ampliación de búsqueda para CategoricalNB: 30 ...


In [16]:
k_values_cat_boundary = [60, 80, 100, 150]
n_bins_values_cat_boundary = [10, 12, 15, 20]

cat_boundary_results = {}

for k in k_values_cat_boundary:
    for n_bins in n_bins_values_cat_boundary:
        nb_cat = Pipeline([
            ("select", SelectKBest(score_func=f_classif, k=k)),
            ("discretizer", KBinsDiscretizer(
                n_bins=n_bins,
                encode="ordinal",
                strategy="quantile"
            )),
            ("classifier", CategoricalNB(alpha=1.0))
        ])
        
        model, scores = evaluate_model(
            name=f"NB-08 Boundary SelectKBest k={k} + quantile bins={n_bins} + CategoricalNB",
            model=nb_cat,
            notes=f"Comprobación del límite de variables para CategoricalNB: {k} variables y {n_bins} bins quantile."
        )
        
        cat_boundary_results[(k, n_bins)] = {
            "model": model,
            "scores": scores
        }

Experimento: NB-08 Boundary SelectKBest k=60 + quantile bins=10 + CategoricalNB
Train accuracy: 0.8430
5-fold CV accuracy: 0.8030 (+/- 0.0157)
Notas: Comprobación del límite de variables para CategoricalNB: 60 variables y 10 bins quantile.
Experimento: NB-08 Boundary SelectKBest k=60 + quantile bins=12 + CategoricalNB
Train accuracy: 0.8490
5-fold CV accuracy: 0.8010 (+/- 0.0073)
Notas: Comprobación del límite de variables para CategoricalNB: 60 variables y 12 bins quantile.
Experimento: NB-08 Boundary SelectKBest k=60 + quantile bins=15 + CategoricalNB
Train accuracy: 0.8590
5-fold CV accuracy: 0.8050 (+/- 0.0134)
Notas: Comprobación del límite de variables para CategoricalNB: 60 variables y 15 bins quantile.
Experimento: NB-08 Boundary SelectKBest k=60 + quantile bins=20 + CategoricalNB
Train accuracy: 0.8700
5-fold CV accuracy: 0.8010 (+/- 0.0146)
Notas: Comprobación del límite de variables para CategoricalNB: 60 variables y 20 bins quantile.


/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/preprocessing/_discretization.py:306: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 42 are removed. Consider decreasing the number of bins.
  warnings.warn(
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/preprocessing/_discretization.py:306: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 17 are removed. Consider decreasing the number of bins.
  warnings.warn(
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/preprocessing/_discretization.py:306: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 43 are removed. Consider decreasing the number of bins.
  warnings.warn(
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-A

Experimento: NB-08 Boundary SelectKBest k=80 + quantile bins=10 + CategoricalNB
Train accuracy: 0.8410
5-fold CV accuracy: 0.8020 (+/- 0.0093)
Notas: Comprobación del límite de variables para CategoricalNB: 80 variables y 10 bins quantile.


/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/preprocessing/_discretization.py:306: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 17 are removed. Consider decreasing the number of bins.
  warnings.warn(
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/preprocessing/_discretization.py:306: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 41 are removed. Consider decreasing the number of bins.
  warnings.warn(
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/preprocessing/_discretization.py:306: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 43 are removed. Consider decreasing the number of bins.
  warnings.warn(
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-A

Experimento: NB-08 Boundary SelectKBest k=80 + quantile bins=12 + CategoricalNB
Train accuracy: 0.8520
5-fold CV accuracy: 0.8010 (+/- 0.0153)
Notas: Comprobación del límite de variables para CategoricalNB: 80 variables y 12 bins quantile.
Experimento: NB-08 Boundary SelectKBest k=80 + quantile bins=15 + CategoricalNB
Train accuracy: 0.8510
5-fold CV accuracy: 0.8010 (+/- 0.0058)
Notas: Comprobación del límite de variables para CategoricalNB: 80 variables y 15 bins quantile.


...

Experimento: NB-08 Boundary SelectKBest k=100 + quantile bins=20 + CategoricalNB
Train accuracy: 0.8730
5-fold CV accuracy: 0.7960 (+/- 0.0066)
Notas: Comprobación del límite de variables para CategoricalNB: 100 variables y 20 bins quantile.
Experimento: NB-08 Boundary SelectKBest k=150 + quantile bins=10 + CategoricalNB
Train accuracy: 0.8560
5-fold CV accuracy: 0.7850 (+/- 0.0148)
Notas: Comprobación del límite de variables para CategoricalNB: 150 variables y 10 bins quantile.


/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/preprocessing/_discretization.py:306: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 27 are removed. Consider decreasing the number of bins.
  warnings.warn(
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/preprocessing/_discretization.py:306: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 42 are removed. Consider decreasing the number of bins.
  warnings.warn(
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/preprocessing/_discretization.py:306: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 43 are removed. Consider decreasing the number of bins.
  warnings.warn(
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-A

Experimento: NB-08 Boundary SelectKBest k=150 + quantile bins=12 + CategoricalNB
Train accuracy: 0.8610
5-fold CV accuracy: 0.7860 (+/- 0.0227)
Notas: Comprobación del límite de variables para CategoricalNB: 150 variables y 12 bins quantile.
Experimento: NB-08 Boundary SelectKBest k=150 + quantile bins=15 + CategoricalNB
Train accuracy: 0.8710
5-fold CV accuracy: 0.7850 (+/- 0.0192)
Notas: Comprobación del límite de variables para CategoricalNB: 150 variables y 15 bins quantile.


/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/preprocessing/_discretization.py:306: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 141 are removed. Consider decreasing the number of bins.
  warnings.warn(
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/preprocessing/_discretization.py:306: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 109 are removed. Consider decreasing the number of bins.
  warnings.warn(
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/preprocessing/_discretization.py:306: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 147 are removed. Consider decreasing the number of bins.
  warnings.warn(
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y

Experimento: NB-08 Boundary SelectKBest k=150 + quantile bins=20 + CategoricalNB
Train accuracy: 0.9000
5-fold CV accuracy: 0.7880 (+/- 0.0189)
Notas: Comprobación del límite de variables para CategoricalNB: 150 variables y 20 bins quantile.


In [17]:
results_df = pd.DataFrame(results)
results_df.sort_values("cv_mean", ascending=False).head(30)

,experiment,train_accuracy,cv_mean,cv_std,notes
77,NB-08 Boundary SelectKBest k=60 + quantile bin...,0.859,0.805,0.013416,Comprobación del límite de variables para Cate...
74,NB-07 Expanded SelectKBest k=60 + quantile bin...,0.859,0.805,0.013416,Ampliación de búsqueda para CategoricalNB: 60 ...
72,NB-07 Expanded SelectKBest k=60 + quantile bin...,0.843,0.803,0.015684,Ampliación de búsqueda para CategoricalNB: 60 ...
75,NB-08 Boundary SelectKBest k=60 + quantile bin...,0.843,0.803,0.015684,Comprobación del límite de variables para Cate...
79,NB-08 Boundary SelectKBest k=80 + quantile bin...,0.841,0.802,0.009274,Comprobación del límite de variables para Cate...
81,NB-08 Boundary SelectKBest k=80 + quantile bin...,0.851,0.801,0.005831,Comprobación del límite de variables para Cate...
78,NB-08 Boundary SelectKBest k=60 + quantile bin...,0.870,0.801,0.014629,Comprobación del límite de variables para Cate...
73,NB-07 Expanded SelectKBest k=60 + quantile bin...,0.849,0.801,0.007348,Ampliación de búsqueda para CategoricalNB: 60 ...
80,NB-08 Boundary SelectKBest k=80 + quantile bin...,0.852,0.801,0.015297,Comprobación del límite de variables para Cate...
76,NB-08 Boundary SelectKBest k=60 + quantile bin...,0.849,0.801,0.007348,Comprobación del límite de variables para Cate...


## Ajustar alfa

In [ ]:
alpha_values = [0.01, 0.05, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0]

alpha_results = {}

for alpha in alpha_values:
    nb_cat_alpha = Pipeline([
        ("select", SelectKBest(score_func=f_classif, k=60)),
        ("discretizer", KBinsDiscretizer(
            n_bins=15,
            encode="ordinal",
            strategy="quantile"
        )),
        ("classifier", CategoricalNB(alpha=alpha))
    ])
    
    model, scores = evaluate_model(
        name=f"NB-09 SelectKBest k=60 + quantile bins=15 + CategoricalNB alpha={alpha}",
        model=nb_cat_alpha,
        notes=f"Ajuste del parámetro de suavizado alpha={alpha} para la mejor configuración de CategoricalNB."
    )
    
    alpha_results[alpha] = {
        "model": model,
        "scores": scores
    }

Experimento: NB-09 SelectKBest k=60 + quantile bins=15 + CategoricalNB alpha=0.01
Train accuracy: 0.8610
5-fold CV accuracy: 0.8110 (+/- 0.0166)
Notas: Ajuste del parámetro de suavizado alpha=0.01 para la mejor configuración de CategoricalNB.
Experimento: NB-09 SelectKBest k=60 + quantile bins=15 + CategoricalNB alpha=0.05
Train accuracy: 0.8610
5-fold CV accuracy: 0.8130 (+/- 0.0163)
Notas: Ajuste del parámetro de suavizado alpha=0.05 para la mejor configuración de CategoricalNB.
Experimento: NB-09 SelectKBest k=60 + quantile bins=15 + CategoricalNB alpha=0.1
Train accuracy: 0.8600
5-fold CV accuracy: 0.8130 (+/- 0.0163)
Notas: Ajuste del parámetro de suavizado alpha=0.1 para la mejor configuración de CategoricalNB.
Experimento: NB-09 SelectKBest k=60 + quantile bins=15 + CategoricalNB alpha=0.5
Train accuracy: 0.8600
5-fold CV accuracy: 0.8110 (+/- 0.0132)
Notas: Ajuste del parámetro de suavizado alpha=0.5 para la mejor configuración de CategoricalNB.
Experimento: NB-09 SelectKBest k

In [20]:
results_df = pd.DataFrame(results)
results_df.sort_values("cv_mean", ascending=False).head(30)

,experiment,train_accuracy,cv_mean,cv_std,notes
92,NB-09 SelectKBest k=60 + quantile bins=15 + Ca...,0.861,0.813,0.016310,Ajuste del parámetro de suavizado alpha=0.05 p...
101,NB-09 SelectKBest k=60 + quantile bins=15 + Ca...,0.860,0.813,0.016310,Ajuste del parámetro de suavizado alpha=0.1 pa...
100,NB-09 SelectKBest k=60 + quantile bins=15 + Ca...,0.861,0.813,0.016310,Ajuste del parámetro de suavizado alpha=0.05 p...
93,NB-09 SelectKBest k=60 + quantile bins=15 + Ca...,0.860,0.813,0.016310,Ajuste del parámetro de suavizado alpha=0.1 pa...
91,NB-09 SelectKBest k=60 + quantile bins=15 + Ca...,0.861,0.811,0.016553,Ajuste del parámetro de suavizado alpha=0.01 p...
102,NB-09 SelectKBest k=60 + quantile bins=15 + Ca...,0.860,0.811,0.013191,Ajuste del parámetro de suavizado alpha=0.5 pa...
99,NB-09 SelectKBest k=60 + quantile bins=15 + Ca...,0.861,0.811,0.016553,Ajuste del parámetro de suavizado alpha=0.01 p...
94,NB-09 SelectKBest k=60 + quantile bins=15 + Ca...,0.860,0.811,0.013191,Ajuste del parámetro de suavizado alpha=0.5 pa...
77,NB-08 Boundary SelectKBest k=60 + quantile bin...,0.859,0.805,0.013416,Comprobación del límite de variables para Cate...
95,NB-09 SelectKBest k=60 + quantile bins=15 + Ca...,0.859,0.805,0.013416,Ajuste del parámetro de suavizado alpha=1.0 pa...


# Discriminant Analysis

## Linear

In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

lda_baseline = LinearDiscriminantAnalysis()

model_lda_baseline, scores_lda_baseline = evaluate_model(
    name="DA-01 LinearDiscriminantAnalysis baseline",
    model=lda_baseline,
    notes="Baseline de LDA. Usa una matriz de covarianza compartida entre clases y genera fronteras lineales."
)

In [3]:
results_df = pd.DataFrame(results)
results_df.sort_values("cv_mean", ascending=False).head(30)

,experiment,train_accuracy,cv_mean,cv_std,notes
0,DA-01 LinearDiscriminantAnalysis baseline,0.953,0.399,0.023108,Baseline de LDA. Usa una matriz de covarianza ...


### LDA con shrinkage automático

In [ ]:
lda_shrink_auto = LinearDiscriminantAnalysis(
    solver="lsqr",
    shrinkage="auto"
)

model_lda_shrink_auto, scores_lda_shrink_auto = evaluate_model(
    name="DA-02 LDA solver=lsqr shrinkage=auto",
    model=lda_shrink_auto,
    notes="LDA con regularización automática de la matriz de covarianza para mejorar estabilidad en alta dimensión."
)

In [7]:
results_df = pd.DataFrame(results)
results_df.sort_values("cv_mean", ascending=False).head(30)

,experiment,train_accuracy,cv_mean,cv_std,notes
1,DA-02 LDA solver=lsqr shrinkage=auto,0.899,0.596,0.020591,LDA con regularización automática de la matriz...
0,DA-01 LinearDiscriminantAnalysis baseline,0.953,0.399,0.023108,Baseline de LDA. Usa una matriz de covarianza ...


### LDA con distintos valores de shrinkage

In [ ]:
shrinkage_values = [0.0, 0.1, 0.2, 0.3, 0.5, 0.7, 0.9, 1.0]

lda_shrinkage_results = {}

for shrinkage in shrinkage_values:
    lda = LinearDiscriminantAnalysis(
        solver="lsqr",
        shrinkage=shrinkage
    )
    
    model, scores = evaluate_model(
        name=f"DA-03 LDA solver=lsqr shrinkage={shrinkage}",
        model=lda,
        notes=f"LDA con shrinkage fijo igual a {shrinkage} para estudiar el efecto de regularizar la covarianza."
    )
    
    lda_shrinkage_results[shrinkage] = {
        "model": model,
        "scores": scores
    }

In [9]:
results_df = pd.DataFrame(results)
results_df.sort_values("cv_mean", ascending=False).head(30)

,experiment,train_accuracy,cv_mean,cv_std,notes
8,DA-03 LDA solver=lsqr shrinkage=0.9,0.785,0.613,0.026571,LDA con shrinkage fijo igual a 0.9 para estudi...
7,DA-03 LDA solver=lsqr shrinkage=0.7,0.814,0.606,0.032465,LDA con shrinkage fijo igual a 0.7 para estudi...
6,DA-03 LDA solver=lsqr shrinkage=0.5,0.834,0.597,0.031401,LDA con shrinkage fijo igual a 0.5 para estudi...
1,DA-02 LDA solver=lsqr shrinkage=auto,0.899,0.596,0.020591,LDA con regularización automática de la matriz...
9,DA-03 LDA solver=lsqr shrinkage=1.0,0.732,0.594,0.027092,LDA con shrinkage fijo igual a 1.0 para estudi...
5,DA-03 LDA solver=lsqr shrinkage=0.3,0.849,0.569,0.033823,LDA con shrinkage fijo igual a 0.3 para estudi...
4,DA-03 LDA solver=lsqr shrinkage=0.2,0.864,0.546,0.036524,LDA con shrinkage fijo igual a 0.2 para estudi...
3,DA-03 LDA solver=lsqr shrinkage=0.1,0.882,0.527,0.027129,LDA con shrinkage fijo igual a 0.1 para estudi...
0,DA-01 LinearDiscriminantAnalysis baseline,0.953,0.399,0.023108,Baseline de LDA. Usa una matriz de covarianza ...
2,DA-03 LDA solver=lsqr shrinkage=0.0,0.953,0.399,0.023108,LDA con shrinkage fijo igual a 0.0 para estudi...


### Ajuste fino de shrinkage

In [ ]:
shrinkage_values_fine = [0.75, 0.8, 0.85, 0.9, 0.95, 1.0]

lda_shrinkage_fine_results = {}

for shrinkage in shrinkage_values_fine:
    lda = LinearDiscriminantAnalysis(
        solver="lsqr",
        shrinkage=shrinkage
    )
    
    model, scores = evaluate_model(
        name=f"DA-04 Fine LDA solver=lsqr shrinkage={shrinkage}",
        model=lda,
        notes=f"Ajuste fino de shrinkage alrededor del mejor valor previo. shrinkage={shrinkage}."
    )
    
    lda_shrinkage_fine_results[shrinkage] = {
        "model": model,
        "scores": scores
    }

In [11]:
results_df = pd.DataFrame(results)
results_df.sort_values("cv_mean", ascending=False).head(30)

,experiment,train_accuracy,cv_mean,cv_std,notes
11,DA-04 Fine LDA solver=lsqr shrinkage=0.8,0.800,0.614,0.025962,Ajuste fino de shrinkage alrededor del mejor v...
12,DA-04 Fine LDA solver=lsqr shrinkage=0.85,0.790,0.614,0.027459,Ajuste fino de shrinkage alrededor del mejor v...
14,DA-04 Fine LDA solver=lsqr shrinkage=0.95,0.777,0.614,0.025962,Ajuste fino de shrinkage alrededor del mejor v...
8,DA-03 LDA solver=lsqr shrinkage=0.9,0.785,0.613,0.026571,LDA con shrinkage fijo igual a 0.9 para estudi...
13,DA-04 Fine LDA solver=lsqr shrinkage=0.9,0.785,0.613,0.026571,Ajuste fino de shrinkage alrededor del mejor v...
10,DA-04 Fine LDA solver=lsqr shrinkage=0.75,0.806,0.611,0.027092,Ajuste fino de shrinkage alrededor del mejor v...
7,DA-03 LDA solver=lsqr shrinkage=0.7,0.814,0.606,0.032465,LDA con shrinkage fijo igual a 0.7 para estudi...
6,DA-03 LDA solver=lsqr shrinkage=0.5,0.834,0.597,0.031401,LDA con shrinkage fijo igual a 0.5 para estudi...
1,DA-02 LDA solver=lsqr shrinkage=auto,0.899,0.596,0.020591,LDA con regularización automática de la matriz...
9,DA-03 LDA solver=lsqr shrinkage=1.0,0.732,0.594,0.027092,LDA con shrinkage fijo igual a 1.0 para estudi...


# Quadratic

In [14]:
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis

qda_baseline = QuadraticDiscriminantAnalysis()

model_qda_baseline, scores_qda_baseline = evaluate_model(
    name="DA-05 QuadraticDiscriminantAnalysis baseline",
    model=qda_baseline,
    notes="Baseline de QDA sin regularización. Estima una matriz de covarianza por clase, lo que puede ser inestable en alta dimensión."
)

/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/discriminant_analysis.py:1024: LinAlgWarning: The covariance matrix of class 0 is not full rank. Increasing the value of parameter `reg_param` might help reducing the collinearity.
  warnings.warn(
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/discriminant_analysis.py:1024: LinAlgWarning: The covariance matrix of class 1 is not full rank. Increasing the value of parameter `reg_param` might help reducing the collinearity.
  warnings.warn(
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/discriminant_analysis.py:1024: LinAlgWarning: The covariance matrix of class 2 is not full rank. Increasing the value of parameter `reg_param` might help reducing the collinearity.
  warnings.warn(
/Users/agostinasquillari/Docume

Experimento: DA-05 QuadraticDiscriminantAnalysis baseline
Train accuracy: 1.0000
5-fold CV accuracy: 0.3300 (+/- 0.0354)
Notas: Baseline de QDA sin regularización. Estima una matriz de covarianza por clase, lo que puede ser inestable en alta dimensión.


/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/discriminant_analysis.py:1024: LinAlgWarning: The covariance matrix of class 0 is not full rank. Increasing the value of parameter `reg_param` might help reducing the collinearity.
  warnings.warn(
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/discriminant_analysis.py:1024: LinAlgWarning: The covariance matrix of class 0 is not full rank. Increasing the value of parameter `reg_param` might help reducing the collinearity.
  warnings.warn(
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/discriminant_analysis.py:1024: LinAlgWarning: The covariance matrix of class 0 is not full rank. Increasing the value of parameter `reg_param` might help reducing the collinearity.
  warnings.warn(
/Users/agostinasquillari/Docume

In [15]:
results_df = pd.DataFrame(results)
results_df.sort_values("cv_mean", ascending=False).head(30)

,experiment,train_accuracy,cv_mean,cv_std,notes
11,DA-04 Fine LDA solver=lsqr shrinkage=0.8,0.800,0.614,0.025962,Ajuste fino de shrinkage alrededor del mejor v...
14,DA-04 Fine LDA solver=lsqr shrinkage=0.95,0.777,0.614,0.025962,Ajuste fino de shrinkage alrededor del mejor v...
12,DA-04 Fine LDA solver=lsqr shrinkage=0.85,0.790,0.614,0.027459,Ajuste fino de shrinkage alrededor del mejor v...
8,DA-03 LDA solver=lsqr shrinkage=0.9,0.785,0.613,0.026571,LDA con shrinkage fijo igual a 0.9 para estudi...
13,DA-04 Fine LDA solver=lsqr shrinkage=0.9,0.785,0.613,0.026571,Ajuste fino de shrinkage alrededor del mejor v...
10,DA-04 Fine LDA solver=lsqr shrinkage=0.75,0.806,0.611,0.027092,Ajuste fino de shrinkage alrededor del mejor v...
7,DA-03 LDA solver=lsqr shrinkage=0.7,0.814,0.606,0.032465,LDA con shrinkage fijo igual a 0.7 para estudi...
6,DA-03 LDA solver=lsqr shrinkage=0.5,0.834,0.597,0.031401,LDA con shrinkage fijo igual a 0.5 para estudi...
1,DA-02 LDA solver=lsqr shrinkage=auto,0.899,0.596,0.020591,LDA con regularización automática de la matriz...
9,DA-03 LDA solver=lsqr shrinkage=1.0,0.732,0.594,0.027092,LDA con shrinkage fijo igual a 1.0 para estudi...


### QDA con reg_param

In [ ]:
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis

reg_values = [0.0, 0.01, 0.05, 0.1, 0.2, 0.3, 0.5, 0.7, 0.9, 1.0]

qda_reg_results = {}

for reg in reg_values:
    qda = QuadraticDiscriminantAnalysis(
        reg_param=reg
    )
    
    model, scores = evaluate_model(
        name=f"DA-06 QDA reg_param={reg}",
        model=qda,
        notes=f"QDA con regularización de covarianza reg_param={reg}."
    )
    
    qda_reg_results[reg] = {
        "model": model,
        "scores": scores
    }

In [17]:
results_df = pd.DataFrame(results)
results_df.sort_values("cv_mean", ascending=False).head(30)

,experiment,train_accuracy,cv_mean,cv_std,notes
11,DA-04 Fine LDA solver=lsqr shrinkage=0.8,0.800,0.614,0.025962,Ajuste fino de shrinkage alrededor del mejor v...
14,DA-04 Fine LDA solver=lsqr shrinkage=0.95,0.777,0.614,0.025962,Ajuste fino de shrinkage alrededor del mejor v...
12,DA-04 Fine LDA solver=lsqr shrinkage=0.85,0.790,0.614,0.027459,Ajuste fino de shrinkage alrededor del mejor v...
8,DA-03 LDA solver=lsqr shrinkage=0.9,0.785,0.613,0.026571,LDA con shrinkage fijo igual a 0.9 para estudi...
13,DA-04 Fine LDA solver=lsqr shrinkage=0.9,0.785,0.613,0.026571,Ajuste fino de shrinkage alrededor del mejor v...
10,DA-04 Fine LDA solver=lsqr shrinkage=0.75,0.806,0.611,0.027092,Ajuste fino de shrinkage alrededor del mejor v...
7,DA-03 LDA solver=lsqr shrinkage=0.7,0.814,0.606,0.032465,LDA con shrinkage fijo igual a 0.7 para estudi...
6,DA-03 LDA solver=lsqr shrinkage=0.5,0.834,0.597,0.031401,LDA con shrinkage fijo igual a 0.5 para estudi...
1,DA-02 LDA solver=lsqr shrinkage=auto,0.899,0.596,0.020591,LDA con regularización automática de la matriz...
15,DA-04 Fine LDA solver=lsqr shrinkage=1.0,0.732,0.594,0.027092,Ajuste fino de shrinkage alrededor del mejor v...


### Aproximación a Diagonal Covariance LDA con escalado

In [18]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

diag_lda_scaled = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LinearDiscriminantAnalysis(
        solver="lsqr",
        shrinkage=1.0
    ))
])

model_diag_lda_scaled, scores_diag_lda_scaled = evaluate_model(
    name="DA-07 StandardScaler + LDA shrinkage=1.0",
    model=diag_lda_scaled,
    notes="Aproximación a una LDA fuertemente regularizada/diagonal, añadiendo escalado previo para comparar estabilidad."
)

/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: Runt

Experimento: DA-07 StandardScaler + LDA shrinkage=1.0
Train accuracy: 0.8010
5-fold CV accuracy: 0.7050 (+/- 0.0114)
Notas: Aproximación a una LDA fuertemente regularizada/diagonal, añadiendo escalado previo para comparar estabilidad.


/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: Runt

In [19]:
results_df = pd.DataFrame(results)
results_df.sort_values("cv_mean", ascending=False).head(30)

,experiment,train_accuracy,cv_mean,cv_std,notes
28,DA-07 StandardScaler + LDA shrinkage=1.0,0.801,0.705,0.011402,Aproximación a una LDA fuertemente regularizad...
11,DA-04 Fine LDA solver=lsqr shrinkage=0.8,0.800,0.614,0.025962,Ajuste fino de shrinkage alrededor del mejor v...
12,DA-04 Fine LDA solver=lsqr shrinkage=0.85,0.790,0.614,0.027459,Ajuste fino de shrinkage alrededor del mejor v...
14,DA-04 Fine LDA solver=lsqr shrinkage=0.95,0.777,0.614,0.025962,Ajuste fino de shrinkage alrededor del mejor v...
8,DA-03 LDA solver=lsqr shrinkage=0.9,0.785,0.613,0.026571,LDA con shrinkage fijo igual a 0.9 para estudi...
13,DA-04 Fine LDA solver=lsqr shrinkage=0.9,0.785,0.613,0.026571,Ajuste fino de shrinkage alrededor del mejor v...
10,DA-04 Fine LDA solver=lsqr shrinkage=0.75,0.806,0.611,0.027092,Ajuste fino de shrinkage alrededor del mejor v...
7,DA-03 LDA solver=lsqr shrinkage=0.7,0.814,0.606,0.032465,LDA con shrinkage fijo igual a 0.7 para estudi...
6,DA-03 LDA solver=lsqr shrinkage=0.5,0.834,0.597,0.031401,LDA con shrinkage fijo igual a 0.5 para estudi...
1,DA-02 LDA solver=lsqr shrinkage=auto,0.899,0.596,0.020591,LDA con regularización automática de la matriz...


### StandardScaler + LDA con distintos shrinkage

In [ ]:
scaled_shrinkage_values = [0.5, 0.7, 0.8, 0.9, 0.95, 1.0]

scaled_lda_results = {}

for shrinkage in scaled_shrinkage_values:
    scaled_lda = Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LinearDiscriminantAnalysis(
            solver="lsqr",
            shrinkage=shrinkage
        ))
    ])
    
    model, scores = evaluate_model(
        name=f"DA-08 StandardScaler + LDA shrinkage={shrinkage}",
        model=scaled_lda,
        notes=f"LDA con escalado previo y shrinkage={shrinkage} para estudiar el efecto conjunto de estandarización y regularización."
    )
    
    scaled_lda_results[shrinkage] = {
        "model": model,
        "scores": scores
    }

In [21]:
results_df = pd.DataFrame(results)
results_df.sort_values("cv_mean", ascending=False).head(30)

,experiment,train_accuracy,cv_mean,cv_std,notes
33,DA-08 StandardScaler + LDA shrinkage=0.95,0.839,0.716,0.013565,LDA con escalado previo y shrinkage=0.95 para ...
32,DA-08 StandardScaler + LDA shrinkage=0.9,0.852,0.708,0.028036,LDA con escalado previo y shrinkage=0.9 para e...
34,DA-08 StandardScaler + LDA shrinkage=1.0,0.801,0.705,0.011402,LDA con escalado previo y shrinkage=1.0 para e...
28,DA-07 StandardScaler + LDA shrinkage=1.0,0.801,0.705,0.011402,Aproximación a una LDA fuertemente regularizad...
31,DA-08 StandardScaler + LDA shrinkage=0.8,0.870,0.672,0.021817,LDA con escalado previo y shrinkage=0.8 para e...
30,DA-08 StandardScaler + LDA shrinkage=0.7,0.877,0.646,0.017436,LDA con escalado previo y shrinkage=0.7 para e...
11,DA-04 Fine LDA solver=lsqr shrinkage=0.8,0.800,0.614,0.025962,Ajuste fino de shrinkage alrededor del mejor v...
14,DA-04 Fine LDA solver=lsqr shrinkage=0.95,0.777,0.614,0.025962,Ajuste fino de shrinkage alrededor del mejor v...
12,DA-04 Fine LDA solver=lsqr shrinkage=0.85,0.790,0.614,0.027459,Ajuste fino de shrinkage alrededor del mejor v...
13,DA-04 Fine LDA solver=lsqr shrinkage=0.9,0.785,0.613,0.026571,Ajuste fino de shrinkage alrededor del mejor v...


### Ajuste fino de shrinkage con StandardScaler

In [22]:
scaled_shrinkage_fine_values = [0.90, 0.925, 0.95, 0.975, 1.0]

scaled_lda_fine_results = {}

for shrinkage in scaled_shrinkage_fine_values:
    scaled_lda = Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LinearDiscriminantAnalysis(
            solver="lsqr",
            shrinkage=shrinkage
        ))
    ])
    
    model, scores = evaluate_model(
        name=f"DA-09 Fine StandardScaler + LDA shrinkage={shrinkage}",
        model=scaled_lda,
        notes=f"Ajuste fino de LDA con escalado y shrinkage={shrinkage}."
    )
    
    scaled_lda_fine_results[shrinkage] = {
        "model": model,
        "scores": scores
    }

/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: Runt

Experimento: DA-09 Fine StandardScaler + LDA shrinkage=0.9
Train accuracy: 0.8520
5-fold CV accuracy: 0.7080 (+/- 0.0280)
Notas: Ajuste fino de LDA con escalado y shrinkage=0.9.


/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: Runt

Experimento: DA-09 Fine StandardScaler + LDA shrinkage=0.925
Train accuracy: 0.8430
5-fold CV accuracy: 0.7150 (+/- 0.0190)
Notas: Ajuste fino de LDA con escalado y shrinkage=0.925.


/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: Runt

Experimento: DA-09 Fine StandardScaler + LDA shrinkage=0.95
Train accuracy: 0.8390
5-fold CV accuracy: 0.7160 (+/- 0.0136)
Notas: Ajuste fino de LDA con escalado y shrinkage=0.95.


/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: Runt

Experimento: DA-09 Fine StandardScaler + LDA shrinkage=0.975
Train accuracy: 0.8340
5-fold CV accuracy: 0.7210 (+/- 0.0162)
Notas: Ajuste fino de LDA con escalado y shrinkage=0.975.


/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: Runt

Experimento: DA-09 Fine StandardScaler + LDA shrinkage=1.0
Train accuracy: 0.8010
5-fold CV accuracy: 0.7050 (+/- 0.0114)
Notas: Ajuste fino de LDA con escalado y shrinkage=1.0.


In [23]:
results_df = pd.DataFrame(results)
results_df.sort_values("cv_mean", ascending=False).head(30)

,experiment,train_accuracy,cv_mean,cv_std,notes
38,DA-09 Fine StandardScaler + LDA shrinkage=0.975,0.834,0.721,0.016248,Ajuste fino de LDA con escalado y shrinkage=0....
33,DA-08 StandardScaler + LDA shrinkage=0.95,0.839,0.716,0.013565,LDA con escalado previo y shrinkage=0.95 para ...
37,DA-09 Fine StandardScaler + LDA shrinkage=0.95,0.839,0.716,0.013565,Ajuste fino de LDA con escalado y shrinkage=0.95.
36,DA-09 Fine StandardScaler + LDA shrinkage=0.925,0.843,0.715,0.018974,Ajuste fino de LDA con escalado y shrinkage=0....
32,DA-08 StandardScaler + LDA shrinkage=0.9,0.852,0.708,0.028036,LDA con escalado previo y shrinkage=0.9 para e...
35,DA-09 Fine StandardScaler + LDA shrinkage=0.9,0.852,0.708,0.028036,Ajuste fino de LDA con escalado y shrinkage=0.9.
39,DA-09 Fine StandardScaler + LDA shrinkage=1.0,0.801,0.705,0.011402,Ajuste fino de LDA con escalado y shrinkage=1.0.
34,DA-08 StandardScaler + LDA shrinkage=1.0,0.801,0.705,0.011402,LDA con escalado previo y shrinkage=1.0 para e...
28,DA-07 StandardScaler + LDA shrinkage=1.0,0.801,0.705,0.011402,Aproximación a una LDA fuertemente regularizad...
31,DA-08 StandardScaler + LDA shrinkage=0.8,0.870,0.672,0.021817,LDA con escalado previo y shrinkage=0.8 para e...


### StandardScaler + PCA + LDA

In [24]:
from sklearn.decomposition import PCA

pca_components_values = [20, 50, 100, 200, 300]

pca_lda_results = {}

for n_components in pca_components_values:
    pca_lda = Pipeline([
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=n_components, random_state=RANDOM_STATE)),
        ("classifier", LinearDiscriminantAnalysis(
            solver="lsqr",
            shrinkage=0.975
        ))
    ])
    
    model, scores = evaluate_model(
        name=f"DA-10 StandardScaler + PCA {n_components} + LDA shrinkage=0.975",
        model=pca_lda,
        notes=f"Reducción de dimensionalidad con PCA a {n_components} componentes antes de LDA regularizado."
    )
    
    pca_lda_results[n_components] = {
        "model": model,
        "scores": scores
    }

/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:335: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:335: RuntimeWarning: overflow encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:335: RuntimeWarning: invalid value encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:336: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = normalizer(A.T @ Q)
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/pyth

Experimento: DA-10 StandardScaler + PCA 20 + LDA shrinkage=0.975
Train accuracy: 0.7540
5-fold CV accuracy: 0.7200 (+/- 0.0084)
Notas: Reducción de dimensionalidad con PCA a 20 componentes antes de LDA regularizado.


/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:335: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:335: RuntimeWarning: overflow encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:335: RuntimeWarning: invalid value encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:336: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = normalizer(A.T @ Q)
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/pyth

Experimento: DA-10 StandardScaler + PCA 50 + LDA shrinkage=0.975
Train accuracy: 0.7540
5-fold CV accuracy: 0.7250 (+/- 0.0071)
Notas: Reducción de dimensionalidad con PCA a 50 componentes antes de LDA regularizado.


/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:335: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:335: RuntimeWarning: overflow encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:335: RuntimeWarning: invalid value encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:336: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = normalizer(A.T @ Q)
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/pyth

...

/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:336: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = normalizer(A.T @ Q)
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:336: RuntimeWarning: overflow encountered in matmul
  Q, _ = normalizer(A.T @ Q)
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:336: RuntimeWarning: invalid value encountered in matmul
  Q, _ = normalizer(A.T @ Q)
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:340: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = qr_normalizer(A @ Q)
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/l

Experimento: DA-10 StandardScaler + PCA 200 + LDA shrinkage=0.975
Train accuracy: 0.7930
5-fold CV accuracy: 0.7210 (+/- 0.0169)
Notas: Reducción de dimensionalidad con PCA a 200 componentes antes de LDA regularizado.


/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:533: RuntimeWarning: divide by zero encountered in matmul
  B = Q.T @ M
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:533: RuntimeWarning: overflow encountered in matmul
  B = Q.T @ M
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:533: RuntimeWarning: invalid value encountered in matmul
  B = Q.T @ M
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:547: RuntimeWarning: divide by zero encountered in matmul
  U = Q @ Uhat
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:547: Run

Experimento: DA-10 StandardScaler + PCA 300 + LDA shrinkage=0.975
Train accuracy: 0.8080
5-fold CV accuracy: 0.7220 (+/- 0.0269)
Notas: Reducción de dimensionalidad con PCA a 300 componentes antes de LDA regularizado.


/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:340: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = qr_normalizer(A @ Q)
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:340: RuntimeWarning: overflow encountered in matmul
  Q, _ = qr_normalizer(A @ Q)
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:340: RuntimeWarning: invalid value encountered in matmul
  Q, _ = qr_normalizer(A @ Q)
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:547: RuntimeWarning: divide by zero encountered in matmul
  U = Q @ Uhat
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9

In [25]:
results_df = pd.DataFrame(results)
results_df.sort_values("cv_mean", ascending=False).head(30)

,experiment,train_accuracy,cv_mean,cv_std,notes
42,DA-10 StandardScaler + PCA 100 + LDA shrinkage...,0.769,0.738,0.019391,Reducción de dimensionalidad con PCA a 100 com...
41,DA-10 StandardScaler + PCA 50 + LDA shrinkage=...,0.754,0.725,0.007071,Reducción de dimensionalidad con PCA a 50 comp...
44,DA-10 StandardScaler + PCA 300 + LDA shrinkage...,0.808,0.722,0.026944,Reducción de dimensionalidad con PCA a 300 com...
43,DA-10 StandardScaler + PCA 200 + LDA shrinkage...,0.793,0.721,0.016852,Reducción de dimensionalidad con PCA a 200 com...
38,DA-09 Fine StandardScaler + LDA shrinkage=0.975,0.834,0.721,0.016248,Ajuste fino de LDA con escalado y shrinkage=0....
40,DA-10 StandardScaler + PCA 20 + LDA shrinkage=...,0.754,0.720,0.008367,Reducción de dimensionalidad con PCA a 20 comp...
33,DA-08 StandardScaler + LDA shrinkage=0.95,0.839,0.716,0.013565,LDA con escalado previo y shrinkage=0.95 para ...
37,DA-09 Fine StandardScaler + LDA shrinkage=0.95,0.839,0.716,0.013565,Ajuste fino de LDA con escalado y shrinkage=0.95.
36,DA-09 Fine StandardScaler + LDA shrinkage=0.925,0.843,0.715,0.018974,Ajuste fino de LDA con escalado y shrinkage=0....
32,DA-08 StandardScaler + LDA shrinkage=0.9,0.852,0.708,0.028036,LDA con escalado previo y shrinkage=0.9 para e...


### Ajuste fino de PCA alrededor de 100

In [26]:
pca_components_fine_values = [75, 100, 125, 150]

pca_lda_fine_results = {}

for n_components in pca_components_fine_values:
    pca_lda = Pipeline([
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=n_components, random_state=RANDOM_STATE)),
        ("classifier", LinearDiscriminantAnalysis(
            solver="lsqr",
            shrinkage=0.975
        ))
    ])
    
    model, scores = evaluate_model(
        name=f"DA-11 Fine StandardScaler + PCA {n_components} + LDA shrinkage=0.975",
        model=pca_lda,
        notes=f"Ajuste fino de PCA alrededor de 100 componentes antes de LDA regularizado."
    )
    
    pca_lda_fine_results[n_components] = {
        "model": model,
        "scores": scores
    }

/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:335: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:335: RuntimeWarning: overflow encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:335: RuntimeWarning: invalid value encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:336: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = normalizer(A.T @ Q)
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/pyth

Experimento: DA-11 Fine StandardScaler + PCA 75 + LDA shrinkage=0.975
Train accuracy: 0.7700
5-fold CV accuracy: 0.7270 (+/- 0.0108)
Notas: Ajuste fino de PCA alrededor de 100 componentes antes de LDA regularizado.


/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:335: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:335: RuntimeWarning: overflow encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:335: RuntimeWarning: invalid value encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:336: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = normalizer(A.T @ Q)
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/pyth

Experimento: DA-11 Fine StandardScaler + PCA 100 + LDA shrinkage=0.975
Train accuracy: 0.7690
5-fold CV accuracy: 0.7380 (+/- 0.0194)
Notas: Ajuste fino de PCA alrededor de 100 componentes antes de LDA regularizado.


/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:335: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:335: RuntimeWarning: overflow encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:335: RuntimeWarning: invalid value encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:336: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = normalizer(A.T @ Q)
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/pyth

Experimento: DA-11 Fine StandardScaler + PCA 125 + LDA shrinkage=0.975
Train accuracy: 0.7790
5-fold CV accuracy: 0.7320 (+/- 0.0169)
Notas: Ajuste fino de PCA alrededor de 100 componentes antes de LDA regularizado.


/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:547: RuntimeWarning: divide by zero encountered in matmul
  U = Q @ Uhat
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:547: RuntimeWarning: overflow encountered in matmul
  U = Q @ Uhat
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:547: RuntimeWarning: invalid value encountered in matmul
  U = Q @ Uhat
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-packages/sklearn/decomposition/_base.py:148: RuntimeWarning: divide by zero encountered in matmul
  X_transformed = X @ self.components_.T
/Users/agostinasquillari/Documents/UPM/Aprendizaje Automatico II/Practica-3y4-AAII/.venv/lib/python3.9/site-package

Experimento: DA-11 Fine StandardScaler + PCA 150 + LDA shrinkage=0.975
Train accuracy: 0.7850
5-fold CV accuracy: 0.7280 (+/- 0.0225)
Notas: Ajuste fino de PCA alrededor de 100 componentes antes de LDA regularizado.


In [27]:
results_df = pd.DataFrame(results)
results_df.sort_values("cv_mean", ascending=False).head(30)

,experiment,train_accuracy,cv_mean,cv_std,notes
46,DA-11 Fine StandardScaler + PCA 100 + LDA shri...,0.769,0.738,0.019391,Ajuste fino de PCA alrededor de 100 componente...
42,DA-10 StandardScaler + PCA 100 + LDA shrinkage...,0.769,0.738,0.019391,Reducción de dimensionalidad con PCA a 100 com...
47,DA-11 Fine StandardScaler + PCA 125 + LDA shri...,0.779,0.732,0.016912,Ajuste fino de PCA alrededor de 100 componente...
48,DA-11 Fine StandardScaler + PCA 150 + LDA shri...,0.785,0.728,0.022494,Ajuste fino de PCA alrededor de 100 componente...
45,DA-11 Fine StandardScaler + PCA 75 + LDA shrin...,0.770,0.727,0.010770,Ajuste fino de PCA alrededor de 100 componente...
41,DA-10 StandardScaler + PCA 50 + LDA shrinkage=...,0.754,0.725,0.007071,Reducción de dimensionalidad con PCA a 50 comp...
44,DA-10 StandardScaler + PCA 300 + LDA shrinkage...,0.808,0.722,0.026944,Reducción de dimensionalidad con PCA a 300 com...
43,DA-10 StandardScaler + PCA 200 + LDA shrinkage...,0.793,0.721,0.016852,Reducción de dimensionalidad con PCA a 200 com...
38,DA-09 Fine StandardScaler + LDA shrinkage=0.975,0.834,0.721,0.016248,Ajuste fino de LDA con escalado y shrinkage=0....
40,DA-10 StandardScaler + PCA 20 + LDA shrinkage=...,0.754,0.720,0.008367,Reducción de dimensionalidad con PCA a 20 comp...
